In [ ]:
# ── Colab Setup (skip automatically if running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # 1. Install NeuralForecast (PyTorch is pre-installed on Colab)
    import subprocess
    subprocess.run(['pip', 'install', 'neuralforecast', '-q'], check=True)

    # 2. Clone the repo to get scripts/models utility code
    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    os.chdir('/content/ECE1508_GenAI/notebooks')

    # 3. Upload data splits — upload train.parquet, val.parquet, test.parquet
    #    from your local  data/splits/  folder when prompted.
    #    If deepar.ipynb already uploaded them this session, skip this step.
    os.makedirs('/content/ECE1508_GenAI/data/splits', exist_ok=True)
    os.makedirs('/content/ECE1508_GenAI/data/predictions', exist_ok=True)

    splits_present = all(
        os.path.exists(f'/content/ECE1508_GenAI/data/splits/{f}')
        for f in ['train.parquet', 'val.parquet', 'test.parquet']
    )
    if not splits_present:
        from google.colab import files as colab_files
        print("Upload train.parquet, val.parquet, and test.parquet from your local data/splits/ folder:")
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            dest = f'/content/ECE1508_GenAI/data/splits/{fname}'
            with open(dest, 'wb') as f:
                f.write(data)
            print(f"  Saved → {dest}")
    else:
        print("Data splits already present — skipping upload.")

    # 4. Verify GPU (strongly recommended — CPU training takes hours)
    import torch
    if torch.cuda.is_available():
        print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    else:
        print("\nNo GPU detected. Go to Runtime → Change runtime type → T4 GPU before running training cells.")

print("Setup complete.")

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MQLoss

from scripts.models.data_loader import load_nf_dataframe, build_full_df, HIST_EXOG_COLS
from scripts.models.metrics import compute_all

plt.rcParams['figure.figsize'] = (14, 4)

CANDIDATES = [24, 60, 120, 240]
FREQ = 1


def patch_params(input_size: int) -> tuple:
    """Return (patch_len, stride) scaled to input_size.

    Ensures at least 4 patches and patch_len <= input_size // 2.
    """
    patch_len = max(4, min(16, input_size // 4))
    stride = max(2, patch_len // 2)
    return patch_len, stride

In [ ]:
extra = HIST_EXOG_COLS  # all 20 non-target feature columns

train_df = load_nf_dataframe('../data/splits/train.parquet', extra_cols=extra)
val_df   = load_nf_dataframe('../data/splits/val.parquet',   extra_cols=extra)
test_df  = load_nf_dataframe('../data/splits/test.parquet',  extra_cols=extra)

trainval_df = build_full_df([train_df, val_df])
full_df     = build_full_df([train_df, val_df, test_df])

print(f"train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")
print(f"hist_exog columns ({len(HIST_EXOG_COLS)}): {HIST_EXOG_COLS}")

In [ ]:
val_maes = {}

for input_size in CANDIDATES:
    patch_len, stride = patch_params(input_size)
    model = PatchTST(
        h=1,
        input_size=input_size,
        patch_len=patch_len,
        stride=stride,
        d_model=128,
        n_heads=8,
        n_layers=3,
        dropout=0.2,
        loss=MQLoss(level=[80, 90]),
        hist_exog_list=HIST_EXOG_COLS,
        max_steps=500,
        early_stop_patience_steps=-1,
        scaler_type='standard',
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(
        df=trainval_df,
        n_windows=None,
        test_size=len(val_df),
        step_size=1,
        refit=False,
    )
    # MQLoss outputs a median column; find it by name
    median_col = [c for c in cv.columns if 'PatchTST' in c and 'median' in c.lower()][0]
    val_mae = float(np.mean(np.abs(cv['y'].values - cv[median_col].values)))
    val_maes[input_size] = val_mae
    print(f"  input_size={input_size:3d}  patch_len={patch_len}  stride={stride}  val MAE={val_mae:.6f}")

best_input_size = min(val_maes, key=val_maes.get)
print(f"\nBest input_size: {best_input_size}  (val MAE={val_maes[best_input_size]:.6f})")

In [ ]:
patch_len, stride = patch_params(best_input_size)

final_model = PatchTST(
    h=1,
    input_size=best_input_size,
    patch_len=patch_len,
    stride=stride,
    d_model=128,
    n_heads=8,
    n_layers=3,
    dropout=0.2,
    loss=MQLoss(level=[80, 90]),
    hist_exog_list=HIST_EXOG_COLS,
    max_steps=1000,
    early_stop_patience_steps=-1,
    scaler_type='standard',
)
nf_final = NeuralForecast(models=[final_model], freq=FREQ)

cv_test = nf_final.cross_validation(
    df=full_df,
    n_windows=None,
    test_size=len(test_df),
    step_size=1,
    refit=False,
)

print(f"Test predictions: {len(cv_test):,} rows")
print(f"Columns: {list(cv_test.columns)}")

In [ ]:
median_col = [c for c in cv_test.columns if 'PatchTST' in c and 'median' in c.lower()][0]
lo80_col   = [c for c in cv_test.columns if 'PatchTST' in c and 'lo-80' in c][0]
hi80_col   = [c for c in cv_test.columns if 'PatchTST' in c and 'hi-80' in c][0]
lo90_col   = [c for c in cv_test.columns if 'PatchTST' in c and 'lo-90' in c][0]
hi90_col   = [c for c in cv_test.columns if 'PatchTST' in c and 'hi-90' in c][0]

y_true = cv_test['y'].values
y_pred = cv_test[median_col].values
lo_80  = cv_test[lo80_col].values
hi_80  = cv_test[hi80_col].values
lo_90  = cv_test[lo90_col].values
hi_90  = cv_test[hi90_col].values

results = compute_all(y_true, y_pred, lo_80, hi_80, lo_90, hi_90)

print("=== PatchTST Test Results ===")
print(f"  RMSE              : {results['rmse']:.6f}")
print(f"  MAE               : {results['mae']:.6f}")
print(f"  Directional Acc   : {results['dir_acc']:.4f}")
print(f"  Coverage 80%      : {results['coverage_80']:.4f}  (target: 0.80)")
print(f"  Coverage 90%      : {results['coverage_90']:.4f}  (target: 0.90)")
print(f"  Sharpe Ratio      : {results['sharpe']:.4f}")
print(f"  Max Drawdown      : {results['max_drawdown']:.6f}")

In [ ]:
n = 200
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(n), y_true[:n], label='Actual return_1h', alpha=0.8, linewidth=0.8, color='black')
ax.plot(range(n), y_pred[:n], label='PatchTST (median)', alpha=0.8, linewidth=0.8, color='darkorange')
ax.fill_between(range(n), lo_90[:n], hi_90[:n], alpha=0.12, color='darkorange', label='90% interval')
ax.fill_between(range(n), lo_80[:n], hi_80[:n], alpha=0.22, color='darkorange', label='80% interval')
ax.axhline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=9)
ax.set_title('PatchTST: predicted vs actual return_1h — first 200 test bars (2024)')
ax.set_xlabel('Test bar index')
ax.set_ylabel('return_1h')
plt.tight_layout()
plt.show()

In [ ]:
test_raw = pd.read_parquet('../data/splits/test.parquet').reset_index(drop=True)

preds_df = pd.DataFrame({
    'ds':       cv_test['ds'].values,
    'datetime': test_raw['datetime'].values[:len(cv_test)],
    'y':        y_true,
    'pred':     y_pred,
    'lo_80':    lo_80,
    'hi_80':    hi_80,
    'lo_90':    lo_90,
    'hi_90':    hi_90,
    'model':    'PatchTST',
})
preds_df.to_parquet('../data/predictions/patchtst_preds.parquet', index=False)
print(f"Saved {len(preds_df):,} rows → data/predictions/patchtst_preds.parquet")
print(preds_df.head(3))